# ARLO Phase 4 Smoke Test

Standalone notebook for compiling the ARLO graph, running a checkpointed test, resuming from the same thread, and inspecting results.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import JSON, Markdown, display

from arlo.runner import compile_for_production, resume_arlo, run_arlo
from arlo.utils.matrix_loader import load_matrix_from_json

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "arlo").exists() and (REPO_ROOT.parent / "arlo").exists():
    REPO_ROOT = REPO_ROOT.parent

REQUIREMENTS_PATH = REPO_ROOT / "data" / "requirements.json"
MATRIX_PATH = REPO_ROOT / "data" / "matrix.json"
CHECKPOINT_DB = REPO_ROOT / "checkpoints" / "arlo.db"

THREAD_ID = "notebook-test-run-1"
RESUME = False

EXPERIMENT_CONFIG = {
    "mode": "stringent",
    "optimizer": "ILP",
    "batch_size": 10,
}

display(Markdown(f"**Thread ID:** `{THREAD_ID}`"))
display(Markdown(f"**Checkpoint DB:** `{CHECKPOINT_DB}`"))

In [ ]:
def create_llm():
    """Instantiate the LLM used by ARLO nodes.

    Replace this placeholder with the project's real LangChain chat model.
    """
    # [MANUAL_INJECTION_REQUIRED: LLM instantiation]
    # INSTRUCTIONS: Replace this with actual LLM instantiation.
    # Example:
    # from langchain_openai import ChatOpenAI
    # return ChatOpenAI(model="gpt-4.1-mini", ...)
    return None


llm = create_llm()
if llm is None:
    raise RuntimeError("LLM not configured. Edit this notebook's LLM injection cell.")

In [ ]:
requirements = None
matrix = None

if not RESUME:
    if not REQUIREMENTS_PATH.exists():
        raise FileNotFoundError(f"Requirements file not found: {REQUIREMENTS_PATH}")
    if not MATRIX_PATH.exists():
        raise FileNotFoundError(f"Matrix file not found: {MATRIX_PATH}")

    requirements = json.loads(REQUIREMENTS_PATH.read_text(encoding="utf-8"))
    matrix = load_matrix_from_json(MATRIX_PATH)

    display(JSON({
        "requirements_count": len(requirements),
        "matrix_rows": len(matrix),
        "matrix_path": str(MATRIX_PATH),
    }))
else:
    display(Markdown("Resume mode enabled. Input files are not loaded."))

In [ ]:
graph = compile_for_production(db_path=CHECKPOINT_DB)
graph

In [ ]:
if RESUME:
    result = resume_arlo(
        graph,
        thread_id=THREAD_ID,
        llm=llm,
    )
else:
    if requirements is None or matrix is None:
        raise RuntimeError("requirements and matrix must be loaded for a new run.")

    result = run_arlo(
        graph,
        requirements=requirements,
        experiment_config=EXPERIMENT_CONFIG,
        matrix=matrix,
        llm=llm,
        thread_id=THREAD_ID,
    )

In [ ]:
summary = {
    "asr_count": len(result.get("asrs", [])),
    "concern_count": len(result.get("concerns", [])),
    "quality_weight_count": len(result.get("quality_weights", {})),
    "stats": result.get("stats", {}),
}

display(JSON(summary))
display(JSON(json.loads(json.dumps(result, default=str))))

In [ ]:
thread_config = {"configurable": {"thread_id": THREAD_ID}}
snapshot = graph.get_state(thread_config)

checkpoint_summary = {
    "next": list(snapshot.next),
    "values_keys": sorted(snapshot.values.keys()),
    "metadata": json.loads(json.dumps(snapshot.metadata, default=str)),
}

display(JSON(checkpoint_summary))